In [ ]:
import numpy as np
import scanpy as sc
import clrmappy as cm

In [ ]:
sc.settings.verbosity = 3
sc.set_figure_params(dpi=80, facecolor="white")

#Setting for umap:
colorspace = 'OKhsl'
min_dist_3d = 0.5
min_dist_2d = 0.5
n_neighbors_2d = 15
n_neighbors_3d = 15
metric_2d = 'euclidean'
metric_3d = 'euclidean'

#settings for filtering
min_genes= 20
max_genes = 200
mt_cutoff = 5
min_cells= 1000

In [ ]:
adata = sc.read_h5ad('')

In [ ]:
adata = cm.preprocess(adata, min_genes=min_genes, max_genes = max_genes, mt_cutoff = 5, min_cells=1000)

In [ ]:

# Compute fresh UMAPs (skip if you already have cached emb2d.npy / emb3d.npy):
umaps = cm.compute_umaps(
    adata,
    min_dist_3d=min_dist_3d, min_dist_2d=min_dist_2d,
    n_neighbors_2d=n_neighbors_2d, n_neighbors_3d=n_neighbors_3d,
    metric_2d=metric_2d, metric_3d=metric_3d,
)
adata = umaps['adata']
emb_2d = umaps['umap_2d']
emb_3d = umaps['umap_3d']


In [ ]:
# OKhsl coloring (3D-based, with saturation optimization + enhancement).
res = cm.emb_to_okhsl(
    emb_3d=emb_3d,
    emb_2d=emb_2d,
    iso_rot_scale=True,
    equal_variance_mode=False,
    pc1_and_2_from_2d=False,
    brightness_range=[0.2, 0.8],
    saturation_enhancement=True,
    saturation_range=[0.1, 1.0],
    center_around='mid',
)
clrmappy_rgb_array = res['clrmappy']
emb_fit = res['emb_fit']

# Alternatives:
# clrmappy_rgb_array = cm.emb_to_rgb(emb_3d)['clrmappy']
# clrmappy_rgb_array = cm.emb_to_cielab(embedding=emb_3d)  # needs R + ucie

In [ ]:
# Interactive 3D scatter of the original 3D UMAP, colored.
cm.plot_emb_3d(
    emb_3d, color=clrmappy_rgb_array,
    title='3D UMAP — OKhsl (unsupervised)',
    dim_labels=('UMAP 1', 'UMAP 2', 'UMAP 3'),
)

In [ ]:
# Fit embedding — the PCA-rotated cloud the OKhsl conversion was applied to.
# Useful for judging how strongly the saturation-optimization algorithm and
# saturation enhancement deformed the original UMAP.
cm.plot_okhsl_fit(emb_fit, color=clrmappy_rgb_array,
                  title='OKhsl fit embedding (PCA-rotated)')

In [ ]:
# 2D UMAP scatter, colored.
cm.plot_emb_2d(
    emb_2d, color=clrmappy_rgb_array,
    title='2D UMAP — OKhsl (unsupervised)',
    dim_labels=('UMAP 1', 'UMAP 2'),
)

In [ ]:
# Side-by-side: OKhsl coloring vs cell-type annotation (Glasbey).
cm.plot_emb_2d_vs_celltype(
    adata, color=clrmappy_rgb_array, celltype_col='class_name',
    title_color='2D UMAP — OKhsl (unsupervised)',
    dim_labels=('UMAP 1', 'UMAP 2'),
)

In [ ]:
# Spatial scatter, colored with OKhsl.
cm.plot_spatial(
    adata, color=clrmappy_rgb_array,
    title='Spatial transcriptomics — OKhsl (unsupervised)',
    dot_size=1,
)

In [ ]:
# Side-by-side spatial: OKhsl coloring vs cell-type annotation (Glasbey).
cm.plot_spatial_vs_celltype(
    adata, color=clrmappy_rgb_array, celltype_col='class_name',
    title_color='Spatial — OKhsl (unsupervised)',
    dot_size=0.75,
)